# Day 1:  Setup + Peek Inside

## Kaggle GPU Setup → Tokenization → Embeddings → Forward Pass

**Duration:** ~2 hours | **GPU Time:** ~30 min | **API Budget:** ~50 requests

Today we'll:
1. Set up Kaggle GPU and verify PyTorch
2. Tokenize text using HuggingFace transformers
3. Explore embeddings and semantic space
4. Trace a complete forward pass through a mini-model
5. Visualize attention patterns

By the end, you'll understand how text becomes predictions.

## Cell 1: Environment Setup & GPU Verification

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Core imports
import torch
import torch.nn as nn
import numpy as np
import math
from pathlib import Path
from datetime import datetime
import json
import time

# Verify GPU
print("=" * 60)
print("🔧 ENVIRONMENT VERIFICATION")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ PyTorch Version: {torch.__version__}")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")
print(f"✓ Device: {device}")

if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()  # Clear any cached memory
    print("✓ Cache cleared")

print("=" * 60)

## Cell 2: Load Tokenizer & Explore Tokenization

In [ ]:
# API call tracker
api_calls = {'total': 0, 'tokenizer_load': 0}

print("\n" + "=" * 60)
print("🔤 TOKENIZATION EXPLORATION")
print("=" * 60)

# Load tokenizer (this makes API calls)
try:
    from transformers import AutoTokenizer
    print("\n→ Loading GPT-2 tokenizer...")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    elapsed = time.time() - start
    api_calls['tokenizer_load'] += 1
    api_calls['total'] += 1
    print(f"✓ Loaded in {elapsed:.2f}s")
except Exception as e:
    print(f"⚠️  Error loading tokenizer: {e}")
    print("Using fallback approach...")

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

# Test tokenization with various texts
test_texts = [
    "Hello, world!",
    "Transformers are amazing.",
    "LLMs can understand complex relationships."
]

print(f"\n→ Tokenizing {len(test_texts)} example texts:")
print("-" * 60)

tokenization_results = []
for text in test_texts:
    token_ids = tokenizer.encode(text)
    token_strings = tokenizer.convert_ids_to_tokens(token_ids)
    
    result = {
        'text': text,
        'token_ids': token_ids,
        'token_strings': token_strings,
        'num_tokens': len(token_ids)
    }
    tokenization_results.append(result)
    
    print(f"\nText: '{text}'")
    print(f"Tokens: {token_strings}")
    print(f"IDs: {token_ids}")
    print(f"Token Count: {len(token_ids)}")

print("\n" + "=" * 60)
print(f"✓ API Calls So Far: {api_calls['total']}/1000")
print("=" * 60)

## Cell 3: Embedding Space Exploration

In [ ]:
print("\n" + "=" * 60)
print("🧠 EMBEDDINGS & SEMANTIC SPACE")
print("=" * 60)

# Create embedding layer
VOCAB_SIZE = 50257  # GPT-2 vocabulary
EMBEDDING_DIM = 768  # Standard hidden size

embedding_layer = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM).to(device)

print(f"\n→ Embedding Layer Configuration:")
print(f"  • Vocabulary Size: {VOCAB_SIZE:,}")
print(f"  • Embedding Dimension: {EMBEDDING_DIM}")
print(f"  • Total Parameters: {VOCAB_SIZE * EMBEDDING_DIM:,}")
print(f"  • Memory: {VOCAB_SIZE * EMBEDDING_DIM * 4 / 1e6:.2f} MB")

# Embed some tokens
sample_tokens = ["Hello", "world", "transformer", "model", "attention"]
token_ids = tokenizer.encode(sample_tokens)
token_ids = torch.tensor([token_ids[:5]]).to(device)

embeddings = embedding_layer(token_ids)

print(f"\n→ Embedding Token: '{sample_tokens}'")
print(f"  • Input Shape: {token_ids.shape}")
print(f"  • Output Shape: {embeddings.shape}")
print(f"  • First embedding (first 10 dims): {embeddings[0, 0, :10].detach().cpu().numpy()}")

# Compute embedding statistics
embeddings_np = embeddings.detach().cpu().numpy()
print(f"\n→ Embedding Statistics:")
print(f"  • Mean: {embeddings_np.mean():.4f}")
print(f"  • Std: {embeddings_np.std():.4f}")
print(f"  • Min: {embeddings_np.min():.4f}")
print(f"  • Max: {embeddings_np.max():.4f}")

# Compute cosine similarity between embeddings
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors"""
    return torch.nn.functional.cosine_similarity(v1, v2, dim=-1).item()

print(f"\n→ Embedding Similarities:")
for i in range(len(sample_tokens)):
    for j in range(i+1, len(sample_tokens)):
        sim = cosine_similarity(embeddings[0, i:i+1], embeddings[0, j:j+1])
        print(f"  • '{sample_tokens[i]}' vs '{sample_tokens[j]}': {sim:.4f}")

print("\n" + "=" * 60)

## Cell 4: Positional Encoding

In [ ]:
print("\n" + "=" * 60)
print("📍 POSITIONAL ENCODING")
print("=" * 60)

def positional_encoding(seq_len, d_model, device):
    """Create positional encodings using sine/cosine pattern"""
    # Position indices
    position = torch.arange(seq_len).unsqueeze(1).float()
    
    # Dimension scaling
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                         (-math.log(10000.0) / d_model))
    
    # Create positional encoding matrix
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe.to(device)

# Generate positional encodings for sequence of length 10
seq_len = 10
pos_enc = positional_encoding(seq_len, EMBEDDING_DIM, device)

print(f"\n→ Positional Encoding:")
print(f"  • Sequence Length: {seq_len}")
print(f"  • Encoding Dimension: {EMBEDDING_DIM}")
print(f"  • Shape: {pos_enc.shape}")

# Show pattern for first few positions
print(f"\n→ Position Encoding Values (first 5 positions, first 10 dims):")
for pos in range(min(5, seq_len)):
    values = pos_enc[pos, :10].detach().cpu().numpy()
    print(f"  Pos {pos}: {values}")

# Add positional encoding to embeddings
embeddings_with_pos = embeddings + pos_enc[:embeddings.shape[1], :].unsqueeze(0)

print(f"\n→ Embeddings with Positional Encoding:")
print(f"  • Original embeddings shape: {embeddings.shape}")
print(f"  • Positional encodings shape: {pos_enc[:embeddings.shape[1], :].unsqueeze(0).shape}")
print(f"  • Combined shape: {embeddings_with_pos.shape}")
print(f"  • Each position has unique information from pos. encoding")

print("\n" + "=" * 60)

## Cell 5: Self-Attention Mechanism

In [ ]:
print("\n" + "=" * 60)
print("⚡ SELF-ATTENTION MECHANISM")
print("=" * 60)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention
    
    Attention(Q, K, V) = Softmax(Q·K^T / √d_k)·V
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply mask if provided (for causal masking)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Apply softmax to get attention weights
    attention_weights = torch.softmax(scores, dim=-1)
    
    # Apply attention to values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Create simple example with 3 tokens
batch_size = 1
seq_len = 3
d_model = 64  # Smaller for visualization

# Create Q, K, V projections
Q = torch.randn(batch_size, seq_len, d_model).to(device)
K = torch.randn(batch_size, seq_len, d_model).to(device)
V = torch.randn(batch_size, seq_len, d_model).to(device)

# Compute attention
attn_output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"\n→ Attention Computation:")
print(f"  • Q shape: {Q.shape}")
print(f"  • K shape: {K.shape}")
print(f"  • V shape: {V.shape}")
print(f"  • Attention scores shape: (batch=1, seq_len=3, seq_len=3)")
print(f"  • Attention weights shape: {attn_weights.shape}")
print(f"  • Output shape: {attn_output.shape}")

print(f"\n→ Attention Weights (how much each position attends to others):")
attn_np = attn_weights[0].detach().cpu().numpy()
print(attn_np)
print("\n  Each row sums to 1.0 (probability distribution)")
print(f"  Row sums: {attn_np.sum(axis=1)}")

# Interpretation
print(f"\n→ Interpretation:")
for i in range(seq_len):
    weights_str = ', '.join([f'Pos {j}: {attn_np[i, j]:.3f}' for j in range(seq_len)])
    print(f"  Token {i} attends to: {weights_str}")

print("\n" + "=" * 60)

## Cell 6: Complete Forward Pass

In [ ]:
print("\n" + "=" * 60)
print("🔄 COMPLETE FORWARD PASS")
print("=" * 60)

class MiniTransformer(nn.Module):
    """Simplified transformer block for educational purposes"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Attention
        self.attention = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        
        # Feed-forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Self-attention with residual connection
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual connection
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        
        return x

# Create mini model
d_model = 256
num_heads = 4
d_ff = 512

transformer_block = MiniTransformer(d_model, num_heads, d_ff).to(device)

print(f"\n→ Mini Transformer Configuration:")
print(f"  • Model Dimension: {d_model}")
print(f"  • Number of Heads: {num_heads}")
print(f"  • Feed-Forward Dimension: {d_ff}")
print(f"  • Total Parameters: {sum(p.numel() for p in transformer_block.parameters()):,}")

# Create input
batch_size = 2
seq_len = 5
x = torch.randn(batch_size, seq_len, d_model).to(device)

# Forward pass
print(f"\n→ Forward Pass:")
print(f"  • Input shape: {x.shape}")

with torch.no_grad():
    output = transformer_block(x)

print(f"  • Output shape: {output.shape}")
print(f"  • Same shape maintained through the block!")
print(f"  • Each token has been enriched with context from attention")

# Statistics
print(f"\n→ Output Statistics:")
output_np = output.detach().cpu().numpy()
print(f"  • Mean: {output_np.mean():.4f}")
print(f"  • Std: {output_np.std():.4f}")
print(f"  • Min: {output_np.min():.4f}")
print(f"  • Max: {output_np.max():.4f}")

print("\n" + "=" * 60)

## Cell 7: End-to-End Text to Prediction

In [ ]:
print("\n" + "=" * 60)
print("🎯 END-TO-END: TEXT → PREDICTION")
print("=" * 60)

# Full pipeline
def text_to_prediction(text, tokenizer, embedding_layer, transformer, output_projection):
    """Complete pipeline: text → tokens → embeddings → transformer → logits"""
    
    # Step 1: Tokenize
    token_ids = tokenizer.encode(text)
    token_ids = torch.tensor([token_ids]).to(device)
    
    # Step 2: Embedding
    embeddings = embedding_layer(token_ids)
    
    # Step 3: Add positional encoding
    pos_enc = positional_encoding(token_ids.shape[1], EMBEDDING_DIM, device)
    embeddings = embeddings + pos_enc.unsqueeze(0)
    
    # Step 4: Transformer block
    with torch.no_grad():
        transformer_output = transformer(embeddings)
    
    # Step 5: Project to vocabulary
    logits = output_projection(transformer_output)
    
    # Step 6: Get probabilities
    probabilities = torch.softmax(logits, dim=-1)
    
    # Step 7: Get next token prediction
    next_token_probs = probabilities[0, -1, :]  # Last token
    next_token_id = torch.argmax(next_token_probs).item()
    next_token_prob = next_token_probs[next_token_id].item()
    
    return {
        'token_ids': token_ids[0].cpu().numpy().tolist(),
        'embeddings_shape': embeddings.shape,
        'transformer_output_shape': transformer_output.shape,
        'logits_shape': logits.shape,
        'next_token_id': next_token_id,
        'next_token': tokenizer.decode([next_token_id]),
        'next_token_prob': next_token_prob,
        'top_5_tokens': torch.topk(next_token_probs, 5)
    }

# Create output projection layer
output_projection = nn.Linear(EMBEDDING_DIM, VOCAB_SIZE).to(device)

# Test with different inputs
test_prompts = [
    "The future of AI is",
    "Hello, my name is",
    "Transformers can be used for"
]

print(f"\n→ Making Predictions:")
for prompt in test_prompts:
    print(f"\n  Input: '{prompt}'")
    
    try:
        result = text_to_prediction(prompt, tokenizer, embedding_layer, transformer_block, output_projection)
        print(f"  • Tokens: {result['token_ids'][:5]}... ({len(result['token_ids'])} total)")
        print(f"  • Predicted next token: '{result['next_token']}' (confidence: {result['next_token_prob']:.2%})")
        
        # Top 5 predictions
        top_ids = result['top_5_tokens'].indices.cpu().numpy()
        top_probs = result['top_5_tokens'].values.cpu().numpy()
        print(f"  • Top 5 predictions:")
        for token_id, prob in zip(top_ids, top_probs):
            token_str = tokenizer.decode([token_id])
            print(f"    - '{token_str}': {prob:.2%}")
    except Exception as e:
        print(f"  ⚠️  Error: {e}")

print("\n" + "=" * 60)

## Cell 8: Summary & Resource Usage

In [ ]:
print("\n" + "=" * 60)
print("📊 WORKSHOP SUMMARY & RESOURCE USAGE")
print("=" * 60)

print("\n✓ What You Learned Today:")
print("  1. Kaggle GPU environment setup and verification")
print("  2. How tokenization converts text → integers")
print("  3. Embeddings: tokens → semantic vectors")
print("  4. Positional encoding: position → information")
print("  5. Self-attention: context → relevance weights")
print("  6. Forward pass: complete text → prediction pipeline")

print("\n📈 Key Insights:")
print("  • Tokenization breaks text into digestible units")
print("  • Embeddings create semantic geometry")
print("  • Attention allows tokens to communicate")
print("  • LayerNorm + residuals stabilize training")
print("  • Complete pipeline is composable and differentiable")

print("\n📊 Resource Usage:")
print(f"  • Total API Calls: {api_calls['total']}/1000 ({api_calls['total']/10:.1f}%)")
print(f"  • Tokenizer Load Calls: {api_calls['tokenizer_load']}")
print(f"  • GPU Time: ~30 minutes (if GPU was used)")
print(f"  • Memory Peak: ~{torch.cuda.max_memory_allocated(device) / 1e9:.2f} GB" if torch.cuda.is_available() else "  • Memory: CPU-based")

print("\n🎯 Next Steps (Day 2):")
print("  • Build complete decoder architecture")
print("  • Explore quantization (INT8, NF4)")
print("  • Compare different model sizes")
print("  • Benchmark inference speed")

print("\n" + "=" * 60)
print("✨ Day 1 Complete! See you tomorrow.")
print("=" * 60)

# Save resource log
resource_log = {
    'timestamp': datetime.now().isoformat(),
    'day': 1,
    'api_calls_total': api_calls['total'],
    'gpu_time_minutes': 30,
    'device': str(device)
}

print(f"\n✓ Resource log saved: {json.dumps(resource_log, indent=2)}")